<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Keyword rules and regular expressions

**[Keyword rules and regular expressions](https://learning.nextia-ai.com/courses/text-routing/m03/keyword-rules-and-regular-expressions/)** · Text Classification and Routing: Send Each Message to the Right Place · Module 3, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** write keyword rules with regular expressions, run Larkfield's rule book on the test
messages, and measure what the rules cost, not only how many messages they catch.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The routes and the cost table, from Module 1. The labelling guidelines, from Module 2. |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m03/keyword-rules-and-regular-expressions/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M03-L01-keyword-rules-and-regular-expressions/keyword-rules-and-regular-expressions-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield inbox of this course (about 9 MB) into a folder `larkfield-inbox/data`, and the course's helper `routing.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 10000 messages (6000 train, 2000 validation, 2000 test), all synthetic.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-inbox` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield inbox is synthetic: we made it for this course. It has no real people. The answers of the hosted models were recorded once, so you need no key.

In [ ]:
# Setup: download the course data into larkfield-inbox/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/"
FILES = {
    "data/corrections.csv": "be3088aecd9b7ba5d07ea222b79e06a476806eba81bc80c2f33da695878e5cc0",
    "data/costs.json": "9a2d566f062c3382b8884ef07756f886179c69a3525d80e3d4c15c0c4fd4507e",
    "data/dataset-card.md": "ec1d7769d1d68a9e51a5b12660364c74352af936df79533a0d2170867171bdd1",
    "data/double_labelled.csv": "ba4a8dadc3c1ba8d48af893124bd33fb7a895f917a2add211486e3dbd353dfc9",
    "data/guidelines.md": "04df0cd72da4e6b7e6aa629ca21205e5395df7b66aae9268618ae14597e45782",
    "data/launch_labels_sample.csv": "600c689325b7c14e56f4a27e92e395eb0b2e965411b1abf9d724720c8b485029",
    "data/launch_month.csv": "7ef3cc2c5fa910023c6330c1e895744358846ce451e551b86be8c9abe1c4289f",
    "data/messages.csv": "266ca23c0551249fc114fc74253103ea8602a04be1fe31d1b7f91bc94ce6dd5e",
    "data/old_queue.csv": "e3cde5d19d3ebe2e6e0e067693c76796c0abbf89df0dc93dc310ad3fe42a5cae",
    "data/predictions/embeddings-newer.csv": "3c7ffe5b5371ca60c3b6bedc17b307142bcfd77f06c97db7ed267b703a50a8d2",
    "data/predictions/embeddings-newer.json": "dcd537f2971b9a48b5354cc9fbb48b87689b83b3a479da9acd119701c2fc19f6",
    "data/predictions/embeddings-older.csv": "a06c490ac17cbc1c3c75380f6cdf7775483de92961cdbb78b2a7914256926539",
    "data/predictions/embeddings-older.json": "db658bbd13861529122807726b6b41e1da9c70e3b55aba55cce9e868a795e8b4",
    "data/predictions/encoder-finetuned.csv": "abced2276671bfd413319573907a48c60952310e50978dba96c0dde0332b7d39",
    "data/predictions/encoder-finetuned.json": "1f7c89271a199c1d659bdc29d8ac0592558a1424218bd0519bbbde9840212b94",
    "data/predictions/gliclass-zero-shot.csv": "2c22e5036de94a376f7c0ccf22603c4d7c624135c8a4965123c9492748814008",
    "data/predictions/gliclass-zero-shot.json": "95ad2acafd48ad2c24516333dba7e78a3eabc7ca616c4053ce2f144f56cbaaec",
    "data/predictions/llm-few-shot.csv": "bcdaee3550f78ebb672465314bbcc9ad26283aaea69265a979739cf6a5f965e2",
    "data/predictions/llm-few-shot.json": "104b446894a0450a7e02fcd07c6db44f3e1a5486002b388374ce6dae4a744a12",
    "data/predictions/llm-zero-shot.csv": "5518e2b668a110b572687313bc13bd689774dff96c0a1614e1cd92e14ddf9ce7",
    "data/predictions/llm-zero-shot.json": "9a72ce600d0d858b00dd3b37247a305f9e8b3fc2fef3b8dbe60c0871b0c1b355",
    "data/predictions/local-llm-prompted.csv": "5bb0d84f8ee9b5086818d3525e1da3f8172b38b7875b10d1d3021b3ad5889f33",
    "data/predictions/local-llm-prompted.json": "ae328379d5ea294b3b59845ee143f24776787e748bc14c165b5e8e9ed4ff7158",
    "data/predictions/local-llm-scores.csv": "f85f570075ea9288fd14a6c3955d4310ba740814f3de472e8a263cd4f9682a5f",
    "data/predictions/local-llm-scores.json": "67a7e07ce065934a947f59c1c36d416eec76f8827f01500ce82ba6e6c4d9b99e",
    "data/predictions/nli-zero-shot.csv": "ecafb001370e5cf2904ce685683beafc86a2c290cd1da80438abb9912d099d3c",
    "data/predictions/nli-zero-shot.json": "54c2e724cb3c855a0fa6302d4ac0ab2df2ffaf788eec4ab198c6feb8d3508928",
    "data/prices.json": "9e98f9056d9d1e53d76a039da9b10d2b4fa54c8a5b35eb6cae32a2abdee7af38",
    "data/recordings/README.md": "d99eb50666045cc2ccf596e100103244fe1513b488fce46e26462a2666053ffc",
    "data/recordings/llm-few-shot.jsonl": "259a93f1d6dcbcb167cd45796c8e09a914e56c7741f14bd0aee53c996e8349d4",
    "data/recordings/llm-labels.jsonl": "cd6c670a23ee78cd3d8b51e5907ebae9d86448aa6236767c11d55227616fa8cc",
    "data/recordings/llm-reasoning.jsonl": "fc8ae631121f3240ce378df429af767ac52ea6e1560aa409de09ffc99b092300",
    "data/recordings/llm-zero-shot.jsonl": "957edfb8811e177c83bf23772f0da561659cf50728ffc95597f85cd7680aed86",
    "data/rules.json": "6f95d5e7b87869ec7975b198a3777cdd96a0db6a869554d922889479c87b2066",
    "routing.py": "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-inbox").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-inbox and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
import csv
with open("data/messages.csv", encoding="utf-8") as f:
    splits = [row["split"] for row in csv.DictReader(f)]
print(f"Ready: {len(splits)} messages ({splits.count('train')} train, {splits.count('validation')} validation, "
      f"{splits.count('test')} test), all synthetic.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course helper as `rt`, reads the three parts of the inbox (`train`, `validation`, `test`), and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the course helper, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import routing as rt
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

messages = pd.read_csv("data/messages.csv")
train = messages[messages["split"] == "train"].reset_index(drop=True)
validation = messages[messages["split"] == "validation"].reset_index(drop=True)
test = messages[messages["split"] == "test"].reset_index(drop=True)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A rule is a pattern

A **regular expression** (regex) is a pattern for text. Python's `re` module finds it in a string. Three pieces
are enough for this lesson:

- `\b` is a word edge: the start or the end of a word.
- `\d` is one digit, and `{6}` means "six times".
- `(a|b)` means `a` or `b`, and `\w*` means "any letters after this".

So `\bLK-\d{6}\b` finds a Larkfield order number, and `\b(refund\w*|return\w*)\b` finds "refund",
"refunded", "return" or "returning".

> **Predict.** Which of the three messages has an order number? Which mentions a refund or a return? Then run the cell.

In [ ]:
import re

samples = ["Where is order LK-123456? Tracking has not moved.",
           "I sent the chair back. When is my refund coming?",
           "Can I return the bench if I don't like the colour?"]
for text in samples:
    order = re.search(r"\bLK-\d{6}\b", text)
    refund = re.search(r"\b(refund\w*|return\w*)\b", text, re.IGNORECASE)
    print(order.group(0) if order else "-", "|", refund.group(0) if refund else "-", "|", text)

> **Check.** The first message has `LK-123456`. The second and the third match the refund pattern. The third is a
> question before buying: by the guidelines, its route is `product`, not `returns_refunds`. The pattern cannot know that.

## 2. Larkfield's rule book

Grace and Priya wrote 11 rules in `data/rules.json`. Each rule has a route, a pattern, a reason (`why`) and a mark
`strong` for the rules that Grace trusts most. The **first** rule that matches gives the route. A message that no
rule matches goes to a person (`review`).

In [ ]:
import json

rules = json.loads(open("data/rules.json", encoding="utf-8").read())["rules"]
for r in rules:
    r["regex"] = re.compile(r["pattern"], re.IGNORECASE)

def first_match(text, rules):
    """The first rule that matches the text, or None."""
    for r in rules:
        if r["regex"].search(text):
            return r
    return None

pd.DataFrame(rules)[["route", "strong", "why"]]

> **Check.** You should see 11 rules. The first three send messages to `needs_person`.

## 3. Run the rules on the test messages

> **Predict.** What share of the 2,000 test messages will match a rule? When a rule matches, how often is it right?

In [ ]:
hits = [first_match(t, rules) for t in test["text"]]
rule_pred = [h["route"] if h else "review" for h in hits]
matched = np.array([h is not None for h in hits])
right = np.array(rule_pred) == test["route"].to_numpy()
print("matched by a rule:", matched.sum(), "of", len(test), "- share", np.round(matched.mean(), 3))
print("right when a rule matched:", right[matched].sum(), "- share", np.round(right[matched].mean(), 3))
print("sent to a person:", (~matched).sum())

> **Check.** You should see 1527 messages matched (share 0.764), 1110 of them right (share 0.727), and 473 sent to a person.

Three in four messages matched, and about three in four of those are right. Is that good? The cost table answers.
`rt.score()` gives the same columns as `route compare`:

In [ ]:
rows = [rt.score("everything to a person", test, ["review"] * len(test)),
        rt.score("rules or a person", test, rule_pred)]
rt.show(rows)

> **Check.** The rules cost 1757.8 dollars per 1,000 messages. Everything to a person costs 1,500.

## 4. Where the cost comes from

Each prediction has a cost (Module 1): 1.50 for a review, 4 for a wrong queue, 20 for an urgent message in a
wrong queue, 200 for a missed account takeover, 40 for a `needs_person` message sent to a team, and 2 for a team
message sent to `needs_person`. The next cell puts each test message in one of these groups.

In [ ]:
def kind(true, pred, priority, topic):
    if pred == true:
        return "right"
    if pred == "review":
        return "review"
    if pred == "needs_person":
        return "team message to a person"
    if topic == "account_takeover":
        return "missed takeover"
    if true == "needs_person":
        return "needs_person message to a team"
    return "urgent wrong queue" if priority == "urgent" else "wrong queue"

test["kind"] = [kind(*x) for x in zip(test["route"], rule_pred, test["priority"], test["topic"])]
test["cost"] = [rt.message_cost(*x) for x in zip(test["route"], rule_pred, test["priority"], test["topic"])]
by_kind = test.groupby("kind").agg(messages=("cost", "size"), dollars=("cost", "sum"))
by_kind["per_1000"] = by_kind["dollars"] / len(test) * 1000
by_kind.sort_values("per_1000", ascending=False)

> **Check.** The largest line is `wrong queue` (634 per 1,000), then `urgent wrong queue` (460) and `review`
> (354.75). Two missed takeovers alone cost 200 per 1,000.

The reviews cost less than a quarter of the total. The rules lose money on the messages that they send to the
**wrong** team with confidence.

## 5. Read the mistakes

Numbers hide the reason. Read five test messages and the rule that caught each one.

In [ ]:
examples = ["M-100017", "M-100056", "M-100318", "M-100516", "M-100053"]
for mid in examples:
    row = test[test["message_id"] == mid].iloc[0]
    h = first_match(row["text"], rules)
    print(f"{mid}  label: {row['route']}  rule: {h['why']} -> {h['route']}  (matched '{h['regex'].search(row['text']).group(0)}')")
    print("   ", row["text"][:160].replace("\n", " "), "\n")

> **Check.** Each rule matched a real word, and each route is wrong. `M-100017` is an account takeover that the
> return rule sent to `returns_refunds`. `M-100056` says *reçu* ("received"), which the billing rule reads as
> "receipt". `M-100318` says that the refund "has nothing to do with it". `M-100516` is about a product called a
> smoker box.

## 6. A word with two meanings

In French, *reçu* means "a receipt", and also "received" (*j'ai reçu*: "I received"). The billing rule has the
word. How often does it decide a test message?

In [ ]:
recu = [(h is not None and h["why"].startswith("invoices") and "reçu" in h["regex"].search(t).group(0).lower())
        for h, t in zip(hits, test["text"])]
caught = test[np.array(recu)]
print(len(caught), "test messages sent to billing because of 'reçu'; their labels:")
print(caught["route"].value_counts().to_dict())

> **Check.** You should see 60 messages, and only 8 of them are billing messages.

A model learns from the labels that *j'ai reçu* goes with other words, such as *mauvais article* ("wrong item").
A rule does only what its pattern says.

## 7. Your turn: a rule for questions before buying

The guidelines say that "Can I return it if I don't like it?" **before buying** is a `product` question. Here is
a rule for it:

```python
policy = {"route": "product", "strong": False, "why": "a question before buying",
          "pattern": r"\b(can i return|may i return|if i (buy|get|order)|before i (buy|order|decide)|je peux le retourner|si je (l'achète|commande))\b"}
```

> **Your turn.** Make a new list `fixed_rules`: a copy of `rules` with `policy` inserted **just before** the first
> rule whose route is `returns_refunds`. Compile its pattern as the others. Then the next cell scores both rule
> books on the **validation** messages, not on the test messages. Why validation? Read the lesson page.

In [ ]:
policy = {"route": "product", "strong": False, "why": "a question before buying",
          "pattern": r"\b(can i return|may i return|if i (buy|get|order)|before i (buy|order|decide)|je peux le retourner|si je (l'achète|commande))\b"}
fixed_rules = ...  # a list: the rules with policy just before the first returns_refunds rule

In [ ]:
if fixed_rules is ...:
    print("Not yet: write fixed_rules in the cell above, then run this cell again.")
else:
    for name, book in (("before", rules), ("after", fixed_rules)):
        pred = [(first_match(t, book) or {"route": "review"})["route"] for t in validation["text"]]
        print(name, "validation cost per 1,000:", round(rt.cost_per_1000(validation, pred), 1))
    val_after = round(rt.cost_per_1000(validation, [(first_match(t, fixed_rules) or {"route": "review"})["route"]
                                                     for t in validation["text"]]), 1)

In [ ]:
expect('the validation cost per 1,000 after your change', val_after if fixed_rules is not ... else ..., 1660.2, tolerance=0.001)

> **Check.** The cost goes from 1677.0 to 1660.2 per 1,000 validation messages. The rule helps a little,
> and the rule book still costs more than a person who reads everything.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m03/keyword-rules-and-regular-expressions/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Text as numbers: bag of words and TF-IDF](https://learning.nextia-ai.com/courses/text-routing/m03/text-as-numbers-tf-idf/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m03/keyword-rules-and-regular-expressions/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.